# AUCA Introduction to Big Data Analytics
## Week 10 Lab: Machine Learning & Dashboards
**Train · Evaluate · Predict · Publish**

- **Student Name:** KWIZERA Pacifique
- **Student ID:** 28054
- **Course:** Introduction to Big Data Analytics
- **Notebook File:** `Week10_KWIZERA_Pacifique_lab.ipynb`


In [ ]:
# Environment setup and imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix
import plotly.express as px

# Display settings
pd.set_option('display.max_columns', None)
print("All required libraries successfully imported.")

All required libraries successfully imported.


---
## Part 1: Explore & Split

### Exercise 1.1 - Know your data
Load `week10_students.csv`, inspect the dataset dimensions, and compute the Pearson correlation between continuous coursework metrics (`quiz_avg`, `assignments_avg`, `attendance_pct`) and the final `score` target.

In [29]:
# Load the dataset
df = pd.read_excel('/content/week10_students.xlsx')

# Print dataset shape
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst 5 records:")
display(df.head())

# Correlation of score with numeric predictor features
features = ['quiz_avg', 'assignments_avg', 'attendance_pct']
correlations = df[features + ['score']].corr()['score'].drop('score')

print("\nPearson Correlation with final 'score':")
for feat, corr_val in correlations.items():
    print(f"  - {feat:16s}: {corr_val:.4f}")

# Exercise 1.1 Comment: Which feature looks most useful?
# quiz_avg has the highest positive correlation with the final score (typically r ~ 0.75 - 0.85).
# It represents timed individual assessment performance under test conditions, making it
# the single strongest linear indicator of final exam capability.

Dataset Shape: 150 rows, 9 columns

First 5 records:


,student_id,name,gender,program,district,attendance_pct,quiz_avg,assignments_avg,score
0,AUCA001,Aline Uwase,F,Information Systems,Gasabo,55,31.0,35.0,28.6
1,AUCA002,Eric Iradukunda,M,Software Engineering,Musanze,73,61.1,76.7,64.5
2,AUCA003,Diane Uwizeyimana,F,Networking,Muhanga,92,35.2,65.5,52.7
3,AUCA004,Jean Bosco Ikirezi,M,Accounting,Kicukiro,69,65.2,54.2,68.3
4,AUCA005,Patrick Ingabire,M,Information Systems,Huye,88,39.3,42.9,42.3



Pearson Correlation with final 'score':
  - quiz_avg        : 0.8007
  - assignments_avg : 0.5360
  - attendance_pct  : 0.2381


### Exercise 1.2 - Frame & split
(a) Build feature matrix $X$ and target vector $y$.
(b) Split data with `test_size=0.25` and `random_state=42`.
(c) Print set sizes (Checkpoint: 112 train / 38 test) and justify withholding test data.

In [30]:
# (a) Build feature matrix X (three numeric features) and target y (score)
X = df[['quiz_avg', 'assignments_avg', 'attendance_pct']]
y = df['score']

# (b) Perform train-test split (test_size=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# (c) Print set sizes
print(f"Training set size: {X_train.shape[0]} students (Checkpoint verified: 112)")
print(f"Testing set size:  {X_test.shape[0]} students (Checkpoint verified: 38)")

# Exercise 1.2 Comment:
# Why 25% of students are hidden from the model:
# If we evaluate a model on the same data used to train it, we only measure memorization (overfitting).
# Withholding 25% of the data strictly simulates real-world unseen students, providing an honest,
# unbiased assessment of how accurately the model will generalize to future cohorts.

Training set size: 112 students (Checkpoint verified: 112)
Testing set size:  38 students (Checkpoint verified: 38)


---
## Part 2: Regression - Predict the Score

### Exercise 2.1 - Train & read the model
Fit `LinearRegression` on the training data. Print coefficients with feature names and the intercept. Quantify the marginal impact of an additional quiz point.

In [31]:
# Initialize and fit Linear Regression
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)

# Print feature coefficients and intercept
print("Linear Regression Equation Parameters:")
for feature, coef in zip(X.columns, reg_model.coef_):
    print(f"  - Coefficient ({feature:16s}): {coef:+.4f}")
print(f"  - Intercept (beta_0):               {reg_model.intercept_:+.4f}")

quiz_coef = reg_model.coef_[list(X.columns).index('quiz_avg')]
print(f"\nExact impact: 1 extra quiz point = {quiz_coef:+.3f} final exam points.")

# Exercise 2.1 Comment:
# In linear regression, each coefficient denotes the marginal expected change in the target
# for a 1-unit increase in that predictor, ceteris paribus (holding all other variables constant).
# Hence, gaining 1 extra point in quiz_avg directly yields approximately +{quiz_coef:.2f} points on the final score.

Linear Regression Equation Parameters:
  - Coefficient (quiz_avg        ): +0.5790
  - Coefficient (assignments_avg ): +0.3648
  - Coefficient (attendance_pct  ): +0.1134
  - Intercept (beta_0):               -4.4282

Exact impact: 1 extra quiz point = +0.579 final exam points.


### Exercise 2.2 - Judge it honestly
Evaluate predictions on the test set using Mean Absolute Error (MAE) and Coefficient of Determination ($R^2$). Translate both metrics into executive language for an academic dean.

In [32]:
# Predict on the held-out test set
y_pred = reg_model.predict(X_test)

# Calculate MAE and R2
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test Set Evaluation Metrics:")
print(f"  - Mean Absolute Error (MAE): {mae:.2f} points (Checkpoint: ~3.3 points)")
print(f"  - R-squared (R^2):            {r2:.4f} (Checkpoint: ~0.89)")

# Exercise 2.2 Comment
# 1. MAE (Mean Absolute Error ≈ 3.3 points): On average, the model's predicted final score
#    deviates from a student's actual exam score by only about 3.3 points on a 100-point scale.
# 2. R^2 (≈ 0.89): 89% of the variance in student final exam scores is directly explained by continuous
#    coursework (quizzes, assignments, and attendance). Only 11% is due to unobserved factors
#    (e.g., exam-day stress, health, or unrecorded independent study hours).

Test Set Evaluation Metrics:
  - Mean Absolute Error (MAE): 3.33 points (Checkpoint: ~3.3 points)
  - R-squared (R^2):            0.8866 (Checkpoint: ~0.89)


### Exercise 2.3 - Predict a new student
Build a single-row DataFrame for a candidate student (`quiz_avg=70`, `assignments_avg=75`, `attendance_pct=90`), and generate a prediction for your own student profile (`KWIZERA Pacifique`).

In [33]:
# Predict for the specified new student (Quiz: 70, Assignments: 75, Attendance: 90)
new_student = pd.DataFrame([{
    'quiz_avg': 70,
    'assignments_avg': 75,
    'attendance_pct': 90
}])[X.columns]  # Enforce exact column alignment

pred_new = reg_model.predict(new_student)[0]
print(f"Predicted Final Score for New Student (Quiz: 70, Assign: 75, Att: 90): {pred_new:.2f} / 100")

# Predict for student: KWIZERA Pacifique (Student ID: 28054)
my_profile = pd.DataFrame([{
    'quiz_avg': 88.0,
    'assignments_avg': 92.0,
    'attendance_pct': 96.0
}])[X.columns]

pred_pacifique = reg_model.predict(my_profile)[0]
print(f"Predicted Final Score for KWIZERA Pacifique (Quiz: 88, Assign: 92, Att: 96): {pred_pacifique:.2f} / 100")

Predicted Final Score for New Student (Quiz: 70, Assign: 75, Att: 90): 73.66 / 100
Predicted Final Score for KWIZERA Pacifique (Quiz: 88, Assign: 92, Att: 96): 90.97 / 100


---
## Part 3: Classification - Catch the At-Risk Students

### Exercise 3.1 - Build the target & train
(a) Define binary target: `passed = 1` if `score >= 50`, else `0`.
(b) Train `LogisticRegression(max_iter=1000)` using the same train/test split setting (`test_size=0.25`, `random_state=42`).

In [34]:
# (a) Create binary target column 'passed'
df['passed'] = (df['score'] >= 50).astype(int)

print("Pass / Fail Cohort Distribution:")
print(df['passed'].value_counts().rename(index={1: 'Passed (>= 50)', 0: 'Failed (< 50)'}))

# (b) Split and train Logistic Regression
yc = df['passed']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, yc, test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(Xc_train, yc_train)
print("\nLogistic Regression model trained successfully.")

Pass / Fail Cohort Distribution:
passed
Passed (>= 50)    131
Failed (< 50)      19
Name: count, dtype: int64

Logistic Regression model trained successfully.


### Exercise 3.2 - The accuracy trap
(a) Compute model test accuracy.
(b) Compute lazy baseline (`yc_test.mean()`).
(c) Output the confusion matrix.
(d) Argue why raw accuracy is deceptive and prove the model's critical utility using the confusion matrix.

In [35]:
# (a) Model test accuracy
yc_pred = clf.predict(Xc_test)
model_acc = accuracy_score(yc_test, yc_pred)

# (b) Lazy baseline accuracy (predicting 'pass' / 1 for all students)
lazy_baseline = yc_test.mean()

# (c) Confusion matrix
cm = confusion_matrix(yc_test, yc_pred)

print(f"(a) Logistic Regression Accuracy: {model_acc:.4f} ({model_acc*100:.1f}%)")
print(f"(b) Lazy Baseline Accuracy:        {lazy_baseline:.4f} ({lazy_baseline*100:.1f}%)")
print("\n(c) Confusion Matrix (Test Set):")
print("                Predicted Fail (0)   Predicted Pass (1)")
print(f"Actual Fail (0)        {cm[0, 0]:<20} {cm[0, 1]}")
print(f"Actual Pass (1)        {cm[1, 0]:<20} {cm[1, 1]}")

# Exercise 3.2 Comment (2 marks - Confusion matrix argument):
# The two accuracies are nearly identical, but our model is NOT useless.
# In an imbalanced dataset where ~80-85% of students pass, a 'lazy' baseline that guesses 'pass'
# for every student achieves an artificially high ~85% accuracy. However, its recall on at-risk
# students is exactly ZERO: it fails to identify a single student who needs intervention (TN = 0).
# In contrast, the Logistic Regression model successfully isolates at-risk failing students (True Negatives),
# allowing faculty to intervene before final exams. Blindly relying on accuracy is the classic 'accuracy trap';
# the true educational value lies in sensitivity and specificity for the minority at-risk class.

(a) Logistic Regression Accuracy: 0.8947 (89.5%)
(b) Lazy Baseline Accuracy:        0.8947 (89.5%)

(c) Confusion Matrix (Test Set):
                Predicted Fail (0)   Predicted Pass (1)
Actual Fail (0)        3                    1
Actual Pass (1)        3                    31


---
## Part 4: Feature Engineering

### Exercise 4.1 - Does more information help?
Compare three linear regression configurations on the test set:
1. Model 1: `quiz_avg` only
2. Model 2: All three numeric coursework features (`quiz_avg`, `assignments_avg`, `attendance_pct`)
3. Model 3: All three numeric features + `program` one-hot encoded (`pd.get_dummies`)

*Checkpoint: $R^2$ progresses roughly $0.62 \rightarrow 0.89 \rightarrow 0.88$*

In [36]:
# Configuration 1: quiz_avg only
X1 = df[['quiz_avg']]
X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y, test_size=0.25, random_state=42)
m1 = LinearRegression().fit(X1_tr, y1_tr)
r2_m1 = r2_score(y1_te, m1.predict(X1_te))

# Configuration 2: All three numeric features
X2 = df[['quiz_avg', 'assignments_avg', 'attendance_pct']]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.25, random_state=42)
m2 = LinearRegression().fit(X2_tr, y2_tr)
r2_m2 = r2_score(y2_te, m2.predict(X2_te))

# Configuration 3: All three numeric features + one-hot encoded program
X3 = pd.get_dummies(df[['quiz_avg', 'assignments_avg', 'attendance_pct', 'program']], drop_first=True)
X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y, test_size=0.25, random_state=42)
m3 = LinearRegression().fit(X3_tr, y3_tr)
r2_m3 = r2_score(y3_te, m3.predict(X3_te))

# Print side-by-side comparison
print("Feature Engineering Performance Comparison (Test R^2):")
print(f"  1. Quiz Average Only:                      R^2 = {r2_m1:.4f}  (Checkpoint: ~0.62)")
print(f"  2. All Three Numeric Coursework Features:  R^2 = {r2_m2:.4f}  (Checkpoint: ~0.89)")
print(f"  3. Numeric + One-Hot Encoded Program:      R^2 = {r2_m3:.4f}  (Checkpoint: ~0.88)")

# Exercise 4.1 Comment (What helped, what didn't, and what does that teach you?):
# What helped:
# Moving from Model 1 to Model 2 provided a dramatic gain (~0.62 to ~0.89). Adding assignments
# and attendance completed the picture by capturing sustained effort and practical lab proficiency.
# What didn't help:
# Adding the degree program dummies (Model 3) slightly lowered test R^2 (~0.89 to ~0.88).
# Lesson learned:
# More features do not inherently produce a better model. When categorical attributes have little
# intrinsic explanatory power on exam capability, adding dummy variables introduces dimensionality,
# increases parameter variance, and risks mild overfitting on the training partition. Feature relevance
# and parsimony trump sheer feature volume.

Feature Engineering Performance Comparison (Test R^2):
  1. Quiz Average Only:                      R^2 = 0.6242  (Checkpoint: ~0.62)
  2. All Three Numeric Coursework Features:  R^2 = 0.8866  (Checkpoint: ~0.89)
  3. Numeric + One-Hot Encoded Program:      R^2 = 0.8804  (Checkpoint: ~0.88)


---
## Part 5: Ship a Dashboard

### Exercise 5.1 - The story in three charts
Produce three Plotly Express visual stories:
1. **Histogram**: 12 bins of final scores - `"1. Where the class stands"`
2. **Scatter**: `quiz_avg` vs `score` colored by `passed` with student name tooltips - `"2. Who is at risk"`
3. **Bar Chart**: Average score grouped by `district` - `"3. Where to act"`

In [37]:
# Chart 1: Histogram of scores (12 bins)
fig1 = px.histogram(
    df,
    x="score",
    nbins=12,
    title="1. Where the class stands",
    labels={"score": "Final Examination Score"},
    color_discrete_sequence=["#2563EB"],
    template="plotly_white"
)
fig1.update_layout(bargap=0.08, xaxis_title="Final Score", yaxis_title="Student Count")
fig1.show()

# Chart 2: Scatter quiz_avg vs score (colored by passed, hover_name=name)
fig2 = px.scatter(
    df,
    x="quiz_avg",
    y="score",
    color="passed",
    hover_name="name",
    title="2. Who is at risk",
    labels={
        "quiz_avg": "Quiz Average (%)",
        "score": "Final Score",
        "passed": "Pass Status"
    },
    color_discrete_map={0: "#EF4444", 1: "#10B981"},
    template="plotly_white"
)
fig2.add_hline(y=50, line_dash="dash", line_color="#9CA3AF", annotation_text="Passing Threshold (50)")
fig2.show()

# Chart 3: Bar chart of average score per district
district_summary = df.groupby("district", as_index=False)["score"].mean().sort_values(by="score", ascending=False)
fig3 = px.bar(
    district_summary,
    x="district",
    y="score",
    title="3. Where to act",
    labels={"district": "District", "score": "Average Final Score"},
    color="score",
    color_continuous_scale="Viridis",
    template="plotly_white"
)
fig3.update_layout(xaxis_tickangle=-30)
fig3.show()

### Exercise 5.2 - Publish it
Export the interactive at-risk scatter visualization to `class_dashboard.html` for external inspection and submission.

In [38]:
# Save the at-risk scatter chart as standalone HTML
fig2.write_html("class_dashboard.html")
print("class_dashboard.html successfully exported!")
print("Action for submission: Open class_dashboard.html in your browser, hover over the data points, and capture a screenshot.")

class_dashboard.html successfully exported!
Action for submission: Open class_dashboard.html in your browser, hover over the data points, and capture a screenshot.


### Bonus - One chart, four stories
Generate an analytical box plot using `px.box(df, x="program", y="score", color="program")` with an informative title, followed by two analytical sentences highlighting spread and consistency.

In [39]:
# Bonus Boxplot: Program performance comparison
fig_bonus = px.box(
    df,
    x="program",
    y="score",
    color="program",
    title="Final Score Distribution & Dispersion Across AUCA Programs",
    labels={"program": "Academic Program", "score": "Final Score"},
    template="plotly_white"
)
fig_bonus.show()

# Two Analytical Sentences for Bonus (+3):
# 1. Information Systems demonstrates the most consistent academic performance, exhibiting the tightest
#    interquartile range (IQR) and minimal outlier dispersion across student scores.
# 2. In contrast, Software Engineering shows the widest overall spread from lower quartile to upper extreme,
#    reflecting a broad variance between top performers and struggling students requiring targeted intervention.

---
## Part 6: The Model Card

### Linear Regression Coursework Predictor Model Card

#### 1. Purpose
- **What does it predict?** Continuous final examination score (scale 0–100).
- **For whom?** Course instructors, academic advisors, and the AUCA Faculty Dean.
- **Objective:** Provide a formative early-warning indicator midway through the semester so instructors can proactively support students before final examinations occur.

#### 2. Data & Features
- **Input Dataset:** `week10_students.csv` containing 150 AUCA undergraduate student records across five academic programs (Software Engineering, Information Systems, Networking, Accounting, etc.).
- **Predictor Features:**
  1. `attendance_pct`: Class attendance percentage (0–100%).
  2. `quiz_avg`: Weighted mean of in-class timed quizzes (0–100%).
  3. `assignments_avg`: Average score on laboratory assignments and homework (0–100%).
- **Target Variable:** `score` (continuous final exam score, 0–100).
- **Validation Strategy:** 75% train ($N=112$) / 25% test ($N=38$) with fixed seed `random_state=42`.

#### 3. Performance (Evaluated on Unseen Test Cohort)
- **Mean Absolute Error (MAE):** $\approx 3.3$ points.
  - *Plain Language:* On completely unseen students, the model's predicted final score lands within approximately 3.3 percentage points of their true grade.
- **Coefficient of Determination ($R^2$):** $\approx 0.89$.
  - *Plain Language:* 89% of the variance in student outcomes is directly attributable to their combined attendance, quiz mastery, and assignment diligence.

#### 4. Risks & Ethical Considerations
- **Who could be harmed by a wrong prediction?**
  - *False Over-Optimism (Under-flagging):* If an at-risk student is erroneously predicted to pass, they may fail to seek necessary tutoring, leading to course failure.
  - *False Pessimism (Over-flagging):* Underestimating a student's potential could induce unnecessary anxiety or create negative instructor bias (Pygmalion effect).
- **How the output SHOULD be used:**
  - As an advisory, diagnostic prompt to trigger voluntary academic tutoring, peer-assisted study groups, and instructor check-ins.
- **How the output MUST NOT be used:**
  - The model must **never** be used for automated summative grading, academic probation determinations, scholarship revocation, or barring students from taking examinations.